##2. Environment Setup

In [1]:
!nvidia-smi

Mon Sep  7 09:27:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
!pip install -q transformers[sentencepiece] datasets sacrebleu rouge_score py7zr

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 492.7/492.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 13.5 MB/s eta 0:00:00


In [4]:
!pip install --upgrade accelerate
!pip uninstall -y transformers accelerate
!pip install transformers accelerate

Found existing installation: transformers 5.16.1
Uninstalling transformers-5.16.1:
  Successfully uninstalled transformers-5.16.1
Found existing installation: accelerate 1.14.0
Uninstalling accelerate-1.14.0:
  Successfully uninstalled accelerate-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 36.1 MB/s eta 0:00:00


##3. Import Libraries

In [5]:
from transformers import pipeline, set_seed
from datasets import load_dataset, load_from_disk
import matplotlib.pyplot as plt
import pandas as pd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

import nltk
from nltk.tokenize import sent_tokenize

from tqdm import tqdm
import torch

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

##4. Load Pre-trained BART Model

In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

In [9]:
model_ckpt = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [11]:
model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(model_ckpt)

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

##5. Load XSum Dataset

In [16]:
dataset_xsum = load_dataset("EdinburghNLP/xsum")

##6. Dataset Exploration

In [17]:
dataset_xsum

DatasetDict({
    train: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 204045
    })
    validation: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11332
    })
    test: Dataset({
        features: ['document', 'summary', 'id'],
        num_rows: 11334
    })
})

In [18]:
dataset_xsum["train"]['document'][1]

'A fire alarm went off at the Holiday Inn in Hope Street at about 04:20 BST on Saturday and guests were asked to leave the hotel.\nAs they gathered outside they saw the two buses, parked side-by-side in the car park, engulfed by flames.\nOne of the tour groups is from Germany, the other from China and Taiwan. It was their first night in Northern Ireland.\nThe driver of one of the buses said many of the passengers had left personal belongings on board and these had been destroyed.\nBoth groups have organised replacement coaches and will begin their tour of the north coast later than they had planned.\nPolice have appealed for information about the attack.\nInsp David Gibson said: "It appears as though the fire started under one of the buses before spreading to the second.\n"While the exact cause is still under investigation, it is thought that the fire was started deliberately."'

In [19]:
dataset_xsum["train"]['summary'][1]

'Two tourist buses have been destroyed by fire in a suspected arson attack in Belfast city centre.'

In [22]:
split_lengths = [len(dataset_xsum[split]) for split in dataset_xsum]

print(f"Split lengths: {split_lengths}")
print(f"Features: {dataset_xsum['train'].column_names}")
print("\nDocument")
print(dataset_xsum["test"][1]["document"])

print("\nSummary")
print(dataset_xsum["test"][1]["summary"])

Split lengths: [204045, 11332, 11334]
Features: ['document', 'summary', 'id']

Document
Officers searched properties in the Waterfront Park and Colonsay View areas of the city on Wednesday.
Detectives said three firearms, ammunition and a five-figure sum of money were recovered.
A 26-year-old man who was arrested and charged appeared at Edinburgh Sheriff Court on Thursday.

Summary
A man has appeared in court after firearms, ammunition and cash were seized by police in Edinburgh.


##7. Data Preprocessing

In [23]:
def convert_examples_to_features(example_batch):
    input_encodings = tokenizer(example_batch['document'], max_length=1024, truncation=True)

    target_encodings = tokenizer(example_batch['summary'], max_length=128, truncation=True)

    return {
        'input_ids': input_encodings['input_ids'],
        'attention_mask': input_encodings['attention_mask'],
        'labels': target_encodings['input_ids']
    }

In [25]:
dataset_xsum_pt = dataset_xsum.map(convert_examples_to_features, batched=True)

Map:   0%|          | 0/204045 [00:00<?, ? examples/s]

Map:   0%|          | 0/11332 [00:00<?, ? examples/s]

Map:   0%|          | 0/11334 [00:00<?, ? examples/s]

In [27]:
dataset_xsum_pt['train']

Dataset({
    features: ['document', 'summary', 'id', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 204045
})

In [28]:
dataset_xsum_pt['train']['input_ids'][1]

[0,
 250,
 668,
 8054,
 439,
 160,
 23,
 5,
 10824,
 9548,
 11,
 8012,
 852,
 23,
 59,
 14722,
 35,
 844,
 28964,
 15,
 378,
 8,
 3958,
 58,
 553,
 7,
 989,
 5,
 2303,
 4,
 50118,
 1620,
 51,
 4366,
 751,
 51,
 794,
 5,
 80,
 8159,
 6,
 9181,
 526,
 12,
 1409,
 12,
 3730,
 11,
 5,
 512,
 2221,
 6,
 20965,
 30,
 8493,
 4,
 50118,
 3762,
 9,
 5,
 2106,
 1134,
 16,
 31,
 1600,
 6,
 5,
 97,
 31,
 436,
 8,
 6951,
 4,
 85,
 21,
 49,
 78,
 363,
 11,
 2874,
 2487,
 4,
 50118,
 133,
 1393,
 9,
 65,
 9,
 5,
 8159,
 26,
 171,
 9,
 5,
 3670,
 56,
 314,
 1081,
 18750,
 15,
 792,
 8,
 209,
 56,
 57,
 4957,
 4,
 50118,
 16991,
 1134,
 33,
 8125,
 5010,
 4055,
 8,
 40,
 1642,
 49,
 2106,
 9,
 5,
 1926,
 3673,
 423,
 87,
 51,
 56,
 1904,
 4,
 50118,
 9497,
 33,
 9826,
 13,
 335,
 59,
 5,
 908,
 4,
 50118,
 43195,
 871,
 9909,
 26,
 35,
 22,
 243,
 2092,
 25,
 600,
 5,
 668,
 554,
 223,
 65,
 9,
 5,
 8159,
 137,
 9592,
 7,
 5,
 200,
 4,
 50118,
 113,
 5771,
 5,
 6089,
 1303,
 16,
 202,
 223,
 803,
 6,
 

In [29]:
dataset_xsum_pt['train']['attention_mask'][1]

[1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 1]

In [31]:
dataset_xsum_pt['train']['labels'][1]

[0,
 9058,
 8376,
 8159,
 33,
 57,
 4957,
 30,
 668,
 11,
 10,
 3986,
 18949,
 908,
 11,
 14837,
 343,
 2100,
 4,
 2]

##8. Model Fine-Tuninig

In [33]:
from transformers import DataCollatorForSeq2Seq

seq2seq_data_collator = DataCollatorForSeq2Seq(tokenizer, model=model_pegasus)

In [35]:
from transformers import TrainingArguments, Trainer

trainer_args = TrainingArguments(
    output_dir='pegasus-xsum', num_train_epochs=1, warmup_steps=500,
    per_device_train_batch_size=1, per_device_eval_batch_size=1,
    weight_decay=0.01, logging_steps=10,
    eval_strategy='steps', eval_steps=500, save_steps=1e6,
    gradient_accumulation_steps=16
)

In [37]:
trainer = Trainer(model=model_pegasus, args=trainer_args,
                  processing_class=tokenizer, data_collator=seq2seq_data_collator,
                  train_dataset=dataset_xsum_pt['test'],
                  eval_dataset=dataset_xsum_pt['validation'])

In [38]:
trainer.train()

Step,Training Loss,Validation Loss
500,30.491229,1.872645
709,29.302786,1.777892


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=709, training_loss=29.713830256495726, metrics={'train_runtime': 4291.0035, 'train_samples_per_second': 2.641, 'train_steps_per_second': 0.165, 'total_flos': 1.0973595509047296e+16, 'train_loss': 29.713830256495726, 'epoch': 1.0})

##9. Model Evaluation

In [40]:
# Evaluation
def generate_batch_sized_chunks(list_of_elements, batch_size):
    """split the dataset into smaller batches that we can process simultaneously
    Yield successive batch-sized chunks from list_of_elements."""
    for i in range(0, len(list_of_elements), batch_size):
        yield list_of_elements[i : i + batch_size]

def calculate_metric_on_test_ds(dataset, metric, model, tokenizer,
                               batch_size=16, device=device,
                               column_text="document",
                               column_summary="summary"):
    article_batches = list(generate_batch_sized_chunks(dataset[column_text], batch_size))
    target_batches = list(generate_batch_sized_chunks(dataset[column_summary], batch_size))

    for article_batch, target_batch in tqdm(
        zip(article_batches, target_batches), total=len(article_batches)):

        inputs = tokenizer(article_batch, max_length=1024,  truncation=True,
                        padding="max_length", return_tensors="pt")

        summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                         attention_mask=inputs["attention_mask"].to(device),
                         length_penalty=0.8, num_beams=8, max_length=128)

        # Finally, we decode the generated texts,
        # replace the token, and add the decoded texts with the references to the metric.
        decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                              clean_up_tokenization_spaces=True)
              for s in summaries]

        decoded_summaries = [d.replace("", " ") for d in decoded_summaries]

        metric.add_batch(predictions=decoded_summaries, references=target_batch)

    # Finally compute and return the ROUGE scores.
    score = metric.compute()
    return score

##10. ROUGE Results

In [44]:
!pip install evaluate
import evaluate


In [42]:
!pip install -U evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00


In [45]:
rouge_names = ["rouge1", "rouge2", "rougeL", "rougeLsum"]

rouge_metric = evaluate.load("rouge")

In [46]:
score = calculate_metric_on_test_ds(
    dataset_xsum['test'][0:10], rouge_metric, trainer.model, tokenizer, batch_size = 2, column_text = 'document', column_summary= 'summary'
)

rouge_dict = dict((rn, score[rn]) for rn in rouge_names )

pd.DataFrame(rouge_dict, index = [f'bart'] )

100%|██████████| 5/5 [00:25<00:00,  5.02s/it]


,rouge1,rouge2,rougeL,rougeLsum
bart,0.014302,0.0,0.014156,0.014131


##11. Model Saving

In [47]:
model_pegasus.save_pretrained("pegasus-xsum")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [48]:
# Save the tokenizer
tokenizer.save_pretrained("tokenizer")

('tokenizer/tokenizer_config.json', 'tokenizer/tokenizer.json')

##12. Text Summarization Example

In [49]:
import torch
from transformers import pipeline

gen_kwargs = {"length_penalty": 0.8, "num_beams":8, "max_length": 128}


sample_text = dataset_xsum["test"][0]["document"]
reference = dataset_xsum["test"][0]["summary"]

# Tokenize the input text
inputs = tokenizer([sample_text], max_length=1024, truncation=True, padding="max_length", return_tensors="pt")

# Move inputs to device
input_ids = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)

# Generate summary using the model's generate method
summaries = model_pegasus.generate(input_ids=input_ids,
                                   attention_mask=attention_mask,
                                   **gen_kwargs)

# Decode the generated summary
decoded_summary = tokenizer.decode(summaries[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)

print("Dialogue:")
print(sample_text)


print("\nReference Summary:")
print(reference)


print("\nModel Summary:")
print(decoded_summary)

Dialogue:
Prison Link Cymru had 1,099 referrals in 2015-16 and said some ex-offenders were living rough for up to a year before finding suitable accommodation.
Workers at the charity claim investment in housing would be cheaper than jailing homeless repeat offenders.
The Welsh Government said more people than ever were getting help to address housing problems.
Changes to the Housing Act in Wales, introduced in 2015, removed the right for prison leavers to be given priority for accommodation.
Prison Link Cymru, which helps people find accommodation after their release, said things were generally good for women because issues such as children or domestic violence were now considered.
However, the same could not be said for men, the charity said, because issues which often affect them, such as post traumatic stress disorder or drug dependency, were often viewed as less of a priority.
Andrew Stevens, who works in Welsh prisons trying to secure housing for prison leavers, said the need for 

##13. Conclusion

### Summary
In this notebook, we fine-tuned the `facebook/bart-large-cnn` model on the **XSum** dataset for abstractive text summarization.

**Key Steps:**
1. **Environment Setup:** Installed necessary libraries like `transformers`, `datasets`, and `evaluate`.
2. **Preprocessing:** Tokenized the XSum dataset, preparing the documents and reference summaries for the model.
3. **Training:** Used the Hugging Face `Trainer` API to fine-tune the model for one epoch.
4. **Evaluation:** Evaluated the model using the ROUGE metric. The model achieved a ROUGE-1 score of approximately 0.014 on a small test sample, which provides a baseline for further optimization.

**Next Steps:**
- Increase the number of training epochs.
- Fine-tune on a larger portion of the training set.
- Experiment with different hyperparameters such as `learning_rate` and `batch_size`.